# iNaturalist Dataset Selection and Extraction

This notebook randomly selects 500 eligible species using seed 42, then extracts 50 `train_mini` images and 10 official validation images per species.


In [ ]:
import json
import random
from collections import defaultdict

import pandas as pd

with open("/content/train_mini.json") as file:
    train_data = json.load(file)

with open("/content/val.json") as file:
    val_data = json.load(file)


def group_by_class(data):
    paths = {}
    for image in data["images"]:
        paths[image["id"]] = image["file_name"]

    groups = defaultdict(list)
    for annotation in data["annotations"]:
        groups[annotation["category_id"]].append(paths[annotation["image_id"]])

    return groups


train_groups = group_by_class(train_data)
val_groups = group_by_class(val_data)

eligible = []
for category_id in train_groups:
    if len(train_groups[category_id]) >= 50 and len(val_groups[category_id]) >= 10:
        eligible.append(category_id)

rng = random.Random(42)
selected = sorted(rng.sample(eligible, 500))

rows = []
for label, category_id in enumerate(selected):
    images = train_groups[category_id].copy()
    rng.shuffle(images)

    for path in images[:50]:
        rows.append(("train_mini", label, category_id, path))

    for path in val_groups[category_id][:10]:
        rows.append(("val", label, category_id, path))

manifest = pd.DataFrame(
    rows,
    columns=["source", "label", "category_id", "file_name"]
)

names = {}
for category in train_data["categories"]:
    names[category["id"]] = category["name"]

class_rows = []
for label, category_id in enumerate(selected):
    class_rows.append((label, category_id, names[category_id]))

classes = pd.DataFrame(
    class_rows,
    columns=["label", "category_id", "name"]
)

manifest.to_csv("extraction_manifest.csv", index=False)
classes.to_csv("selected_classes.csv", index=False)

print(manifest["source"].value_counts())
print("Classes:", len(classes))


In [ ]:
!wget -c https://ml-inat-competition-datasets.s3.amazonaws.com/2021/train_mini.tar.gz
!wget -c https://ml-inat-competition-datasets.s3.amazonaws.com/2021/val.tar.gz


In [ ]:
import tarfile
from pathlib import Path

manifest = pd.read_csv("extraction_manifest.csv")
output = Path("selected_images")
output.mkdir(exist_ok=True)


def extract_selected(archive_name, selected_paths):
    selected_paths = set(selected_paths)

    with tarfile.open(archive_name, "r:gz") as archive:
        for member in archive:
            path = member.name.removeprefix("./")

            if member.isfile() and path in selected_paths:
                archive.extract(member, output, filter="data")


extract_selected(
    "train_mini.tar.gz",
    manifest[manifest["source"] == "train_mini"]["file_name"]
)

extract_selected(
    "val.tar.gz",
    manifest[manifest["source"] == "val"]["file_name"]
)

number_of_images = sum(path.is_file() for path in output.rglob("*"))
print(number_of_images)


In [ ]:
import shutil

shutil.make_archive(
    "selected_images",
    "gztar",
    root_dir="selected_images"
)
